In [28]:
import pandas as pd
import numpy as np
from pathlib import Path
import re
import time

prereq = "para trabajar pre req progv22025-09-15095342_procesado.xlsx"
historial = "detalle matricula sistemas pedacito calcs.xlsx"
                                

In [29]:
class AnalizadorPrerrequisitos:
    def __init__(self):
        self.df_prereq = None
        self.df_historial = None
        self.resultado = None
        self.max_niveles_cadena = 5
        # Cache para optimización
        self.cache_materias_estudiante = {}
        self.cache_prerrequisitos = {}
    
    def cargar_documentos(self):
        """Solicita y carga los documentos de Excel"""
        print("=== ANALIZADOR OPTIMIZADO DE PRERREQUISITOS CON CADENAS ===\n")
        
        # Solicitar archivo de prerrequisitos
        while True:
            try:
                ruta_prereq = prereq #input("Ingrese la ruta del archivo 'pre_requisitos.xlsx': ").strip()
                self.df_prereq = pd.read_excel(ruta_prereq)
                print(f"✓ Archivo de prerrequisitos cargado: {len(self.df_prereq)} registros")
                break
            except Exception as e:
                print(f"Error al cargar prerrequisitos: {e}")
                print("Intente nuevamente.\n")
        
        # Solicitar archivo de historial
        while True:
            try:
                ruta_historial = historial#input("Ingrese la ruta del archivo 'historial_asignaturas.xlsx': ").strip()
                self.df_historial = pd.read_excel(ruta_historial)
                print(f"✓ Archivo de historial cargado: {len(self.df_historial)} registros")
                break
            except Exception as e:
                print(f"Error al cargar historial: {e}")
                print("Intente nuevamente.\n")
    
    def validar_columnas_y_datos(self):
        """Valida que existan las columnas necesarias y verifica duplicados"""
        print("Validando estructura de datos...")
        
        # Columnas requeridas en prerrequisitos
        cols_prereq_req = ['Smbarul_Key_Rule', 'Programa']
        cols_prereq_faltantes = [col for col in cols_prereq_req if col not in self.df_prereq.columns]
        
        if cols_prereq_faltantes:
            raise ValueError(f"Faltan columnas en prerrequisitos: {cols_prereq_faltantes}")
        
        # Columnas requeridas en historial
        cols_hist_req = ['Codigo_Programa', 'Cod materia curso', 'Pidm', 'Materia_Aprobada', 'Calificacion_Final']
        cols_hist_faltantes = [col for col in cols_hist_req if col not in self.df_historial.columns]
        
        # Verificar si existe columna Periodo
        if 'Periodo' not in self.df_historial.columns:
            print("⚠️  ADVERTENCIA: No se encontró la columna 'Periodo' en el historial.")
            print("Se procederá sin validación de duplicados por periodo.")
            self.validar_duplicados_sin_periodo()
        else:
            self.validar_duplicados_con_periodo()
        
        if cols_hist_faltantes:
            raise ValueError(f"Faltan columnas en historial: {cols_hist_faltantes}")
        
        print("✓ Validación completada")
    
    def validar_duplicados_con_periodo(self):
        """Valida duplicados usando Pidm + Periodo + Cod materia curso"""
        print("- Validando duplicados con columna Periodo...")
        
        # Verificar duplicados en la combinación única
        combinacion_unica = ['Pidm', 'Periodo', 'Cod materia curso']
        duplicados = self.df_historial.duplicated(subset=combinacion_unica, keep=False)
        
        if duplicados.any():
            registros_duplicados = self.df_historial[duplicados]
            print(f"\n⚠️  ENCONTRADOS {duplicados.sum()} REGISTROS DUPLICADOS:")
            print("Registros con la misma combinación Pidm + Periodo + Cod materia curso:")
            
            for _, grupo in registros_duplicados.groupby(combinacion_unica):
                print(f"- Pidm: {grupo.iloc[0]['Pidm']}, Periodo: {grupo.iloc[0]['Periodo']}, "
                      f"Materia: {grupo.iloc[0]['Cod materia curso']} ({len(grupo)} registros)")
            
            print(f"\nEliminando {duplicados.sum()} registros duplicados...")
            self.df_historial = self.df_historial[~duplicados].copy()
            print(f"Registros restantes: {len(self.df_historial)}")
    
    def validar_duplicados_sin_periodo(self):
        """Información sobre duplicados cuando no existe la columna Periodo"""
        print("- Analizando repeticiones de Pidm + Cod materia curso...")
        
        combinaciones = self.df_historial.groupby(['Pidm', 'Cod materia curso']).size()
        repeticiones = combinaciones[combinaciones > 1]
        
        if len(repeticiones) > 0:
            print(f"ℹ️  Se encontraron {len(repeticiones)} combinaciones estudiante-materia con múltiples registros.")
            print("Esto es normal ya que los estudiantes pueden tomar la misma materia varias veces.")
            print(f"Total de registros extra por repeticiones: {repeticiones.sum() - len(repeticiones)}")
        else:
            print("ℹ️  No se encontraron repeticiones en Pidm + Cod materia curso.")
    
    def obtener_columnas_opciones(self):
        """Obtiene las columnas de opciones de prerrequisitos (Opcion_Prereq_1 a Opcion_Prereq_21)"""
        patron = r'^Opcion_Prereq_\d+$'
        columnas_opciones = [col for col in self.df_prereq.columns if re.match(patron, col)]
        columnas_opciones.sort(key=lambda x: int(x.split('_')[-1]))
        return columnas_opciones
    
    def parsear_prerrequisito(self, prereq_str):
        """Parsea una cadena de prerrequisito y devuelve una lista de códigos de asignaturas"""
        if pd.isna(prereq_str) or str(prereq_str).strip() == '':
            return []
        
        prereq_str = str(prereq_str).strip()
        
        if '&' in prereq_str:
            codigos = [codigo.strip() for codigo in prereq_str.split('&')]
            return [codigo for codigo in codigos if codigo]
        else:
            return [prereq_str] if prereq_str else []
    
    def preparar_datos_optimizados(self):
        """Prepara estructuras de datos optimizadas para el procesamiento"""
        print("Preparando estructuras de datos optimizadas...")
        
        # 1. Crear tabla de historial con información agregada por estudiante-materia
        print("- Agregando información de historial por estudiante-materia...")
        
        # Función auxiliar para obtener nota final
        def obtener_nota_final(serie):
            df = serie.to_frame()
             # Reconstruir el grupo completo usando el índice
            grupo_completo = self.df_historial.loc[df.index]

            aprobados = grupo_completo[grupo_completo['Materia_Aprobada'].astype(str).str.upper() == 'Y']
            if not aprobados.empty:
                return aprobados.iloc[0]['Calificacion_Final']  # primera aprobación
            else:
                return grupo_completo.iloc[-1]['Calificacion_Final']  # último intento
        
        # Función auxiliar para verificar si aprobó
        def verificar_aprobacion(serie):
            return (serie.astype(str).str.upper() == 'Y').any()
        
        historial_agg = self.df_historial.groupby(['Pidm', 'Cod materia curso']).agg({
            'Materia_Aprobada': verificar_aprobacion,
            'Calificacion_Final': obtener_nota_final,
            'Codigo_Programa': 'first'
        }).reset_index()
        
        historial_agg.columns = ['Pidm', 'Cod materia curso', 'Aprobada', 'Nota_Final', 'Codigo_Programa']
        
        # Contar intentos
        intentos = self.df_historial.groupby(['Pidm', 'Cod materia curso']).size().reset_index(name='Intentos')
        historial_agg = historial_agg.merge(intentos, on=['Pidm', 'Cod materia curso'])
        
        self.historial_optimizado = historial_agg
        
        # 2. Preparar tabla de prerrequisitos melted
        print("- Procesando opciones de prerrequisitos...")
        columnas_opciones = self.obtener_columnas_opciones()
        
        prereq_melted = []
        for _, row in self.df_prereq.iterrows():
            for i, col in enumerate(columnas_opciones, 1):
                if pd.notna(row[col]) and str(row[col]).strip():
                    prereq_melted.append({
                        'Smbarul_Key_Rule': row['Smbarul_Key_Rule'],
                        'Programa': row['Programa'],
                        'Opcion_Num': i,
                        'Prerrequisitos_Raw': str(row[col]).strip(),
                        'Prerrequisitos_Codigos': self.parsear_prerrequisito(row[col])
                    })
        
        self.prereq_melted = pd.DataFrame(prereq_melted)
        print(f"✓ Estructuras optimizadas creadas. Opciones de prerrequisitos: {len(self.prereq_melted)}")
    
    def verificar_opciones_prerrequisitos_vectorizado(self, estudiantes_materias):
        """Verifica todas las opciones de prerrequisitos de forma vectorizada"""
        if estudiantes_materias.empty:
            return pd.DataFrame()
            
        print(f"- Verificando cumplimiento de opciones de prerrequisitos para {len(estudiantes_materias)} combinaciones...")
        
        # Merge con opciones de prerrequisitos
        datos_con_prereq = estudiantes_materias.merge(
            self.prereq_melted, 
            left_on='Cod materia curso', 
            right_on='Smbarul_Key_Rule', 
            how='left'
        )
        
        # Filtrar solo registros que tienen prerrequisitos
        datos_con_prereq = datos_con_prereq.dropna(subset=['Opcion_Num'])
        
        if datos_con_prereq.empty:
            return pd.DataFrame()
        
        # Expandir prerrequisitos por estudiante y opción
        filas_expandidas = []
        for _, row in datos_con_prereq.iterrows():
            prerrequisitos = row['Prerrequisitos_Codigos']
            if prerrequisitos and len(prerrequisitos) > 0:
                for codigo_prereq in prerrequisitos:
                    filas_expandidas.append({
                        'Pidm': row['Pidm'],
                        'Materia_Principal': row['Cod materia curso'],
                        'Opcion_Num': row['Opcion_Num'],
                        'Codigo_Prereq': codigo_prereq,
                        'Programa': row['Programa']
                    })
        
        if not filas_expandidas:
            return pd.DataFrame()
            
        df_expandido = pd.DataFrame(filas_expandidas)
        
        # Hacer merge con historial para verificar cumplimiento
        cumplimiento = df_expandido.merge(
            self.historial_optimizado[['Pidm', 'Cod materia curso', 'Aprobada', 'Nota_Final', 'Intentos']],
            left_on=['Pidm', 'Codigo_Prereq'],
            right_on=['Pidm', 'Cod materia curso'],
            how='left'
        )
        
        # Marcar como no cumplido si no se encuentra o no está aprobada
        cumplimiento['Cumple'] = cumplimiento['Aprobada'].fillna(False)
        cumplimiento['Encontrada'] = cumplimiento['Aprobada'].notna()
        
        # Agregar por estudiante, materia y opción para determinar si toda la opción se cumple
        opciones_cumplimiento = cumplimiento.groupby(['Pidm', 'Materia_Principal', 'Opcion_Num']).agg({
            'Cumple': 'all',  # Todos los prerrequisitos deben cumplirse
            'Encontrada': 'all',  # Todos deben encontrarse
            'Codigo_Prereq': lambda x: list(x),
            'Nota_Final': lambda x: list(x),
            'Intentos': lambda x: list(x)
        }).reset_index()
        
        # Para cada estudiante-materia, encontrar la primera opción que cumple
        resultado_final = []
        
        for (pidm, materia), grupo in opciones_cumplimiento.groupby(['Pidm', 'Materia_Principal']):
            # Ordenar por número de opción y buscar la primera que cumple
            grupo_sorted = grupo.sort_values('Opcion_Num')
            
            opcion_elegida = None
            for _, opcion in grupo_sorted.iterrows():
                if opcion['Cumple'] and opcion['Encontrada']:
                    opcion_elegida = opcion
                    break
            
            if opcion_elegida is not None:
                resultado_final.append({
                    'Pidm': pidm,
                    'Cod materia curso': materia,
                    'Opcion_Elegida': opcion_elegida['Opcion_Num'],
                    'Prerrequisitos_Codigos': opcion_elegida['Codigo_Prereq'],
                    'Prerrequisitos_Notas': opcion_elegida['Nota_Final'],
                    'Prerrequisitos_Intentos': opcion_elegida['Intentos'],
                    'Estado': 'Prerrequisito cumplido'
                })
            else:
                # No se encontró opción que cumpla
                primera_opcion = grupo_sorted.iloc[0]
                resultado_final.append({
                    'Pidm': pidm,
                    'Cod materia curso': materia,
                    'Opcion_Elegida': primera_opcion['Opcion_Num'],
                    'Prerrequisitos_Codigos': primera_opcion['Codigo_Prereq'],
                    'Prerrequisitos_Notas': primera_opcion['Nota_Final'],
                    'Prerrequisitos_Intentos': primera_opcion['Intentos'],
                    'Estado': 'Tiene prerrequisito y no se encontró en la historia'
                })
        
        return pd.DataFrame(resultado_final)
    
    # Reemplaza este método en tu clase
    def construir_cadenas_por_nivel(self, df_base, nivel=1, visited=None):
        """
        Construye la lista de prerrequisitos por niveles, acumulando todos los niveles
        en un DataFrame con columnas:
        ['Pidm', 'Materia_Original', 'Cod materia curso', 'Nivel', 'Posicion',
        'Nota_Nivel', 'Intentos_Nivel', 'Estado_Nivel']
        Evita re-procesar la misma combinación (Pidm, Materia_Original, codigo).
        """
        if visited is None:
            visited = set()

        if nivel > self.max_niveles_cadena or df_base is None or df_base.empty:
            return pd.DataFrame()

        filas_nivel = []

        for _, row in df_base.iterrows():
            pidm = row['Pidm']
            # Materia_Original: si viene (de niveles previos) la usamos; si no, la materia principal es 'Cod materia curso'
            materia_original = row.get('Materia_Original', row.get('Cod materia curso'))
            prerrequisitos_codigos = row.get('Prerrequisitos_Codigos', []) or []
            prerrequisitos_notas = row.get('Prerrequisitos_Notas', []) or []
            prerrequisitos_intentos = row.get('Prerrequisitos_Intentos', []) or []

            if not isinstance(prerrequisitos_codigos, list) or len(prerrequisitos_codigos) == 0:
                continue

            # Normalizar longitudes
            if not isinstance(prerrequisitos_notas, list):
                prerrequisitos_notas = [None] * len(prerrequisitos_codigos)
            if not isinstance(prerrequisitos_intentos, list):
                prerrequisitos_intentos = [None] * len(prerrequisitos_codigos)

            while len(prerrequisitos_notas) < len(prerrequisitos_codigos):
                prerrequisitos_notas.append(None)
            while len(prerrequisitos_intentos) < len(prerrequisitos_codigos):
                prerrequisitos_intentos.append(None)

            for i, codigo in enumerate(prerrequisitos_codigos):
                if codigo is None:
                    continue
                # evitar NaNs numéricos que se conviertan a string raro
                if isinstance(codigo, float) and np.isnan(codigo):
                    continue

                key = (pidm, materia_original, str(codigo))
                if key in visited:
                    continue
                visited.add(key)

                filas_nivel.append({
                    'Pidm': pidm,
                    'Materia_Original': materia_original,
                    'Cod materia curso': codigo,
                    'Nivel': nivel,
                    'Posicion': i + 1,
                    'Nota_Nivel': prerrequisitos_notas[i] if i < len(prerrequisitos_notas) else None,
                    'Intentos_Nivel': prerrequisitos_intentos[i] if i < len(prerrequisitos_intentos) else None,
                    'Estado_Nivel': 'Aprobada' if (i < len(prerrequisitos_notas) and pd.notna(prerrequisitos_notas[i])) else 'No Aprobada'
                })

        if not filas_nivel:
            return pd.DataFrame()

        df_nivel = pd.DataFrame(filas_nivel)

        # Buscar prerrequisitos del siguiente nivel (usando los códigos actuales como "materia principal")
        siguiente_nivel = self.verificar_opciones_prerrequisitos_vectorizado(
            df_nivel[['Pidm', 'Cod materia curso']].drop_duplicates()
        )

        if siguiente_nivel.empty:
            return df_nivel

        # Asociar Materia_Original para que los registros siguientes sepan a qué materia principal pertenecen
        mapa = df_nivel[['Pidm', 'Cod materia curso', 'Materia_Original']].drop_duplicates(subset=['Pidm', 'Cod materia curso'])
        siguiente_nivel = siguiente_nivel.merge(
            mapa,
            left_on=['Pidm', 'Cod materia curso'],
            right_on=['Pidm', 'Cod materia curso'],
            how='left'
        )

        # Recursión: procesar siguiente nivel y concatenar resultados (manteniendo visited)
        deeper = self.construir_cadenas_por_nivel(siguiente_nivel, nivel + 1, visited)

        if deeper is None or deeper.empty:
            return df_nivel

        return pd.concat([df_nivel, deeper], ignore_index=True)
    
    
	# Reemplaza TODO el método procesar_prerrequisitos por este
    def procesar_prerrequisitos(self):
        """Procesa todos los registros de forma optimizada (versión que agrega cadenas)"""
        print("\n=== PROCESANDO PRERREQUISITOS CON OPTIMIZACIÓN VECTORIZADA ===")
        inicio = time.time()

        self.validar_columnas_y_datos()
        self.preparar_datos_optimizados()

        # 1. Procesar prerrequisitos directos
        estudiantes_materias = self.df_historial[['Pidm', 'Cod materia curso']].drop_duplicates()
        print(f"Procesando {len(estudiantes_materias)} combinaciones únicas estudiante-materia...")

        # Verificar prerrequisitos directos (esto devolverá por materia principal las opciones y sus códigos)
        prereq_directos = self.verificar_opciones_prerrequisitos_vectorizado(estudiantes_materias)

        # 2. Construir cadenas completas (si hay prerrequisitos directos)
        if not prereq_directos.empty:
            print("Construyendo cadenas completas de prerrequisitos (niveles)...")
            cadenas_completas = self.construir_cadenas_por_nivel(prereq_directos.copy())
        else:
            print("No se encontraron prerrequisitos directos para procesar cadenas.")
            cadenas_completas = pd.DataFrame()

        # 3. Reconstruir resultado final
        print("Reconstruyendo datos finales...")
        self.resultado = self.df_historial.copy()

        # Agregar información de intentos de materia principal
        self.resultado = self.resultado.merge(
            self.historial_optimizado[['Pidm', 'Cod materia curso', 'Intentos']].rename(columns={'Intentos': 'Intentos_Materia_Principal'}),
            on=['Pidm', 'Cod materia curso'],
            how='left'
        )

        # Si hay prerrequisitos (directos o en cadena), agregarlos de forma agregada por materia principal
        if not prereq_directos.empty:
            # Si hay cadenas, agrupamos por (Pidm, Materia_Original) para obtener listas ordenadas por nivel y posición
            if not cadenas_completas.empty:
                agrupado = (
                    cadenas_completas
                    .sort_values(['Nivel', 'Posicion'])
                    .groupby(['Pidm', 'Materia_Original'])
                    .agg({
                        'Cod materia curso': lambda x: list(x),
                        'Nota_Nivel': lambda x: list(x),
                        'Intentos_Nivel': lambda x: list(x),
                        'Nivel': lambda x: list(x)
                    })
                    .reset_index()
                )
                # Renombrar columnas para que coincidan con el esquema usado en expandir_prerrequisitos_a_columnas
                agrupado = agrupado.rename(columns={
                    'Materia_Original': 'Cod materia curso',
                    'Cod materia curso': 'Prerrequisitos_Codigos',
                    'Nota_Nivel': 'Prerrequisitos_Notas',
                    'Intentos_Nivel': 'Prerrequisitos_Intentos',
                    'Nivel': 'Prerrequisitos_Niveles'
                })
                agrupado['Prerrequisitos_Tipos'] = agrupado['Prerrequisitos_Niveles'].apply(
                    lambda L: ['Directo' if n == 1 else 'Cadena' for n in L]
                )
            else:
                # No se encontraron cadenas, usamos los prerrequisitos directos tal cual
                agrupado = prereq_directos[['Pidm', 'Cod materia curso', 'Prerrequisitos_Codigos', 'Prerrequisitos_Notas', 'Prerrequisitos_Intentos']].copy()
                agrupado = agrupado.rename(columns={'Cod materia curso': 'Cod materia curso'})
                agrupado['Prerrequisitos_Niveles'] = agrupado['Prerrequisitos_Codigos'].apply(
                    lambda x: [1] * len(x) if isinstance(x, list) else []
                )
                agrupado['Prerrequisitos_Tipos'] = agrupado['Prerrequisitos_Niveles'].apply(
                    lambda L: ['Directo' for _ in L]
                )

            # Añadir columna Estado/Opcion_Elegida de prereq_directos para la materia principal si existe
            estado_info = prereq_directos[['Pidm', 'Cod materia curso', 'Estado', 'Opcion_Elegida']].copy()
            # Agrupado usa 'Cod materia curso' como la materia principal (ya renombrada)
            agrupado = agrupado.merge(estado_info, on=['Pidm', 'Cod materia curso'], how='left')

            # Merge final en resultado
            self.resultado = self.resultado.merge(
                agrupado,
                on=['Pidm', 'Cod materia curso'],
                how='left'
            )

            # Expandir prerrequisitos a columnas separadas
            self.expandir_prerrequisitos_a_columnas()
        else:
            self.resultado['Estado'] = 'No tiene prerrequisito'

        # Llenar valores faltantes
        self.resultado['Estado'] = self.resultado['Estado'].fillna('No tiene prerrequisito')
        self.resultado['Intentos_Materia_Principal'] = self.resultado['Intentos_Materia_Principal'].fillna(1)

        tiempo_total = time.time() - inicio
        print(f"✓ Procesamiento completado en {tiempo_total:.2f} segundos")
        print(f"Velocidad: {len(self.resultado)/tiempo_total:.0f} registros/segundo")

    
    
# Reemplaza este método en tu clase
    def expandir_prerrequisitos_a_columnas(self):
        """Expande las listas de prerrequisitos a columnas separadas (solo hasta lo necesario).
        Además agrega nivel y tipo por prerequisito.
        """
        print("Expandiendo prerrequisitos a columnas separadas...")

        def safe_get_list_item(lista, index):
            if isinstance(lista, list) and len(lista) > index:
                return lista[index]
            return None

        # Determinar el máximo real de prerrequisitos en todas las filas
        if 'Prerrequisitos_Codigos' in self.resultado.columns:
            max_prereq = int(self.resultado['Prerrequisitos_Codigos'].dropna().apply(lambda x: len(x) if isinstance(x, list) else 0).max() or 0)
        else:
            max_prereq = 0

        # Crear dinámicamente columnas por cada prerrequisito (codigo, nota, intentos, nivel, tipo, es_cadena)
        for i in range(max_prereq):
            self.resultado[f'Prereq_{i+1}_Codigo'] = self.resultado['Prerrequisitos_Codigos'].apply(lambda x: safe_get_list_item(x, i) if isinstance(x, list) else None)
            self.resultado[f'Prereq_{i+1}_Nota'] = self.resultado['Prerrequisitos_Notas'].apply(lambda x: safe_get_list_item(x, i) if isinstance(x, list) else None)
            self.resultado[f'Prereq_{i+1}_Intentos'] = self.resultado['Prerrequisitos_Intentos'].apply(lambda x: safe_get_list_item(x, i) if isinstance(x, list) else None)
            self.resultado[f'Prereq_{i+1}_Nivel'] = self.resultado['Prerrequisitos_Niveles'].apply(lambda x: safe_get_list_item(x, i) if isinstance(x, list) else None)
            self.resultado[f'Prereq_{i+1}_Tipo'] = self.resultado['Prerrequisitos_Tipos'].apply(lambda x: safe_get_list_item(x, i) if isinstance(x, list) else None)
            self.resultado[f'Prereq_{i+1}_EsCadena'] = self.resultado[f'Prereq_{i+1}_Tipo'].apply(lambda t: True if t == 'Cadena' else (False if pd.notna(t) else None))

        # Contar prerrequisitos totales (flatten)
        def count_prerequisites(lista):
            if isinstance(lista, list):
                return len(lista)
            return 0

        self.resultado['Num_Prerrequisitos_Total'] = self.resultado['Prerrequisitos_Codigos'].apply(count_prerequisites)

        # Contar prerrequisitos directos (nivel == 1)
        def count_directs(niveles):
            if isinstance(niveles, list):
                return sum(1 for n in niveles if n == 1)
            return 0

        self.resultado['Num_Prerrequisitos_Directos'] = self.resultado.get('Prerrequisitos_Niveles', []).apply(lambda x: count_directs(x) if isinstance(x, list) else 0)

        # Opcional: eliminar columnas agregadas temporales si ya no las quieres
        columnas_a_eliminar = ['Prerrequisitos_Codigos', 'Prerrequisitos_Notas', 'Prerrequisitos_Intentos', 'Prerrequisitos_Niveles', 'Prerrequisitos_Tipos']
        for col in columnas_a_eliminar:
            if col in self.resultado.columns:
                self.resultado = self.resultado.drop(columns=[col])


    
    def generar_reporte(self):
        """Genera un reporte resumen del análisis"""
        if self.resultado is None:
            print("No hay resultados para reportar")
            return
        
        print("\n=== REPORTE DE RESULTADOS ===")
        
        resumen = self.resultado['Estado'].value_counts()
        print("\nResumen por estado:")
        for estado, cantidad in resumen.items():
            porcentaje = (cantidad / len(self.resultado)) * 100
            print(f"- {estado}: {cantidad} ({porcentaje:.1f}%)")
        
        if 'Num_Prerrequisitos_Directos' in self.resultado.columns:
            print(f"\nEstadísticas de prerrequisitos directos:")
            prereq_stats = self.resultado['Num_Prerrequisitos_Directos'].value_counts().sort_index()
            for num, cantidad in prereq_stats.items():
                print(f"- {num} prerrequisitos: {cantidad} materias")
        
        if 'Intentos_Materia_Principal' in self.resultado.columns:
            intentos_promedio = self.resultado['Intentos_Materia_Principal'].mean()
            print(f"\nPromedio de intentos por materia: {intentos_promedio:.2f}")
        
        print(f"\nTotal de registros analizados: {len(self.resultado)}")
    
    def guardar_resultados(self):
        """Guarda los resultados en un archivo Excel"""
        if self.resultado is None:
            print("No hay resultados para guardar")
            return
        
        timestamp = pd.Timestamp.now().strftime("%Y-%m-%d%H%M%S")
        nombre_archivo = "resultados_prerrequisitos_cadenas_"+timestamp+".xlsx"
        
        try:
            # Limpiar columnas completamente vacías
            self.resultado = self.resultado.dropna(axis=1, how='all')
            
            self.resultado.to_excel(nombre_archivo, index=False)
            print(f"✓ Resultados guardados en: {nombre_archivo}")
            print(f"Columnas en el archivo: {len(self.resultado.columns)}")
        except Exception as e:
            print(f"Error al guardar resultados: {e}")
    
    def ejecutar(self):
        """Ejecuta el análisis completo optimizado"""
        try:
            self.cargar_documentos()
            self.procesar_prerrequisitos()
            self.generar_reporte()
            self.guardar_resultados()
            print("\n¡Análisis optimizado completado exitosamente!")
        except Exception as e:
            print(f"\nError durante la ejecución: {e}")
            import traceback
            traceback.print_exc()

# Función principal
def main():
    analizador = AnalizadorPrerrequisitos()
    analizador.ejecutar()

if __name__ == "__main__":
    main()

=== ANALIZADOR OPTIMIZADO DE PRERREQUISITOS CON CADENAS ===

✓ Archivo de prerrequisitos cargado: 3101 registros
✓ Archivo de historial cargado: 2707 registros

=== PROCESANDO PRERREQUISITOS CON OPTIMIZACIÓN VECTORIZADA ===
Validando estructura de datos...
- Validando duplicados con columna Periodo...
✓ Validación completada
Preparando estructuras de datos optimizadas...
- Agregando información de historial por estudiante-materia...
- Procesando opciones de prerrequisitos...
✓ Estructuras optimizadas creadas. Opciones de prerrequisitos: 2574
Procesando 2332 combinaciones únicas estudiante-materia...
- Verificando cumplimiento de opciones de prerrequisitos para 2332 combinaciones...


C:\Users\vittorinor\AppData\Local\Temp\ipykernel_23024\1165750014.py:225: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  cumplimiento['Cumple'] = cumplimiento['Aprobada'].fillna(False)


Construyendo cadenas completas de prerrequisitos (niveles)...
- Verificando cumplimiento de opciones de prerrequisitos para 1571 combinaciones...


C:\Users\vittorinor\AppData\Local\Temp\ipykernel_23024\1165750014.py:225: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  cumplimiento['Cumple'] = cumplimiento['Aprobada'].fillna(False)


- Verificando cumplimiento de opciones de prerrequisitos para 505 combinaciones...
Reconstruyendo datos finales...
Expandiendo prerrequisitos a columnas separadas...
✓ Procesamiento completado en 6.35 segundos
Velocidad: 426 registros/segundo

=== REPORTE DE RESULTADOS ===

Resumen por estado:
- No tiene prerrequisito: 1460 (53.9%)
- Prerrequisito cumplido: 804 (29.7%)
- Tiene prerrequisito y no se encontró en la historia: 443 (16.4%)

Estadísticas de prerrequisitos directos:
- 0 prerrequisitos: 1460 materias
- 1 prerrequisitos: 683 materias
- 2 prerrequisitos: 564 materias

Promedio de intentos por materia: 1.38

Total de registros analizados: 2707
✓ Resultados guardados en: resultados_prerrequisitos_cadenas_2025-09-17164859.xlsx
Columnas en el archivo: 180

¡Análisis optimizado completado exitosamente!
